In [ ]:
%run helpers.ipynb
from sympy import factorial

In [2]:
# To do: check equality of parents for many methods
class T_symb_basis_elt:
    def __init__(self,str_rep,wght,parent):
        self.heis_dim=parent.heis_dim
        self.parent=parent
        self.heis_dim=parent.heis_dim
        self.str_rep=str_rep
        
        self.vec_rep=[0]*(self.heis_dim+4)
        basis_str_list=['Y','H','E','X']
        for i in range(1,self.heis_dim):
            basis_str_list.append('e%d'%i)
        basis_str_list.append('N')
        self.vec_rep[basis_str_list.index(str_rep)]=1
        
        self.wght=wght
        self.ad_dict={}
        self.dual_ad_dict={}
        self.cochain_ad_dicts={}
        self.ext_ad_dicts={}
        
    def iprod(self,other):
        return self.parent.elt(self.vec_rep).iprod(other)
    
    def __eq__(self,other): 
        if type(other)==int:
            return False
        if self.parent!=other.parent: return False
        return self.vec_rep==other.vec_rep
        
    def __str__(self):
        return self.str_rep
    
    def __repr__(self):
        return self.str_rep
    
    def __lt__(self,other):
        if self.parent!=other.parent: raise invalid_parent_exception 
        return self.parent.basis.index(self)<self.parent.basis.index(other)
    
    def __gt__(self,other):
        if self.parent!=other.parent: raise invalid_parent_exception 
        return self.parent.basis.index(self)>self.parent.basis.index(other)
    
    def __le__(self,other):
        if self.parent!=other.parent: raise invalid_parent_exception 
        return self.parent.basis.index(self)<=self.parent.basis.index(other)

    def __ge__(self,other):
        if self.parent!=other.parent: raise invalid_parent_exception 
        return self.parent.basis.index(self)>=self.parent.basis.index(other)
    
    def __add__(self,other):
        if type(other) in [T_symb_elt,T_symb_basis_elt]:
            if self.parent!=other.parent: raise invalid_parent_exception 
            result=[other.vec_rep[i] for i in range(len(other.vec_rep))]
            result[self.parent.basis.index(self)]+=1
            return T_symb_elt(result,self.parent)
        if other==0:
            return self
        raise ValueError('Only objs of types T_symb_elt and T_symb_basis_elt can be added to an obj of type T_symb_elt')
    
    def __radd__(self,other):
        return self+other
    
    def __neg__(self):
        result=[0]*len(self.parent.basis)
        result[self.parent.basis.index(self)]=-1
        return T_symb_elt(result,self.parent)
    
    def __sub__(self,other):
        return self+(-other)
    
    def __mul__(self,other):
        result=[0]*len(self.parent.basis)
        result[self.parent.basis.index(self)]=other
        return T_symb_elt(result,self.parent)
    
    def __rmul__(self,other):
        return(self*other)
    
    def __getstate__(self):
        return self.__dict__
    
    def __setstate__(self,d):
        self.__dict__=d
        
    def ad(self,other,dual=False):
        return self.parent.elt(self.vec_rep).ad(other,dual)
    
    def cast_as_ext_elt(self):
        return T_symb_elt(self.vec_rep,self.parent).cast_as_ext_elt()
    
    def cast_as_cochain(self):
        return T_symb_elt(self.vec_rep,self.parent).cast_as_cochain()

In [19]:
class T_symb_elt:
    
    def __init__(self,vec_rep,parent):
        '''vec_rep: a list of length len(self.basis) with integer entries'''
        self.parent=parent
        self.basis=parent.basis
        self.vec_rep=list(vec_rep)
        self.heis_dim=parent.heis_dim
        
    
    def __str__(self):
        if self.vec_rep==[0]*len(self.basis):
            return '0'
        
        result=''
        cntr=0
        while result=='':
            if self.vec_rep[cntr]!=0:
                if self.vec_rep[cntr]==1:
                    result=str(self.basis[cntr])
                elif self.vec_rep[cntr]==-1:
                    result='-'+str(self.basis[cntr])
                elif type(self.vec_rep[cntr])==Add:
                    result='('+str(self.vec_rep[cntr])+')*'+str(self.basis[cntr])
                else:
                    result = str(self.vec_rep[cntr])+'*'+str(self.basis[cntr])
            cntr+=1
        for i in range(cntr,len(self.basis)):
            if self.vec_rep[i]==1:
                result+=' + '+str(self.basis[i])
            elif self.vec_rep[i]==-1:
                result+=' - '+str(self.basis[i])
            elif self.vec_rep[i]!=0:
                if type(self.vec_rep[i])==Add: c_str='('+str(self.vec_rep[i])+')*'
                else: c_str=str(self.vec_rep[i])
                result+=' + '+c_str+'*'+str(self.basis[i])
        return result

    def __repr__(self):
        return str(self)
    
    def __eq__(self,other):
        if other==0 and type(other)==int:
            return self.vec_rep==[0]*(len(self.basis))
        if not hasattr(other,'parent'): return False
        if self.parent!=other.parent: return False
        return self.vec_rep==other.vec_rep
    
    def __neg__(self):
        return(T_symb_elt([-A for A in self.vec_rep],self.parent))
    
    def __add__(self,other):
        if other==0:
            return self
        return T_symb_elt([self.vec_rep[i]+other.vec_rep[i] 
                           for i in range(len(self.basis))],self.parent)   
    
    def __radd__(self,other):
        return self+other
    
    def __sub__(self,other):
        return self+(-other)
            
    def __mul__(self,other):
        return T_symb_elt([other*A for A in self.vec_rep],self.parent)
    
    def __rmul__(self,other):
        return self*other
    
    def __getstate__(self):
        return self.__dict__
    
    def __setstate__(self,d):
        self.__dict__=d
        
    def ad(self,t,dual=False):
        '''returns: a T_symb_elt object representing ad(se,t)
           if dual=True, the object returned represents ad(se,t*)'''
        P=self.parent
        E=self.parent.ext_alg
        C=self.parent.cochain_complex
        
        if type(t)==T_symb_elt or type(t)==T_symb_basis_elt:
            if t.parent!=P: raise invalid_parent_exception
            
            if self==0 or t==0:
                return P.elt([0]*len(P.basis))

            result=P.elt([0]*len(P.basis))
            for j in range(len(P.basis)):
                if t.vec_rep[j]!=0:
                    for i in range(len(P.basis)):
                        if self.vec_rep[i]!=0: 
                            coeff=self.vec_rep[i]*t.vec_rep[j]
                            if dual: d=P.basis[i].dual_ad_dict
                            else: d=P.basis[i].ad_dict
                            result+=coeff*d[P.basis_strs[j]]
            return result
        
        if type(t)==ext_elt:
            if t.parent!=E: raise invalid_parent_exception
            if t==0 or self==0: return E.elt({})
            
            result=E.elt({})
            for k in t.coeff_dict:
                deg=E.deg(E.elt({k:1}))
                for i in range(len(P.basis)):
                    if self.vec_rep[i]!=0:
                        result+=self.vec_rep[i]*t.coeff_dict[k]*P.basis[i].ext_ad_dicts[deg][k]
            return result
        
        if type(t)==cochain:
            if t.parent!=C: raise invalid_parent_exception
            if t==0 or self==0: return C.elt({})
            
            result=C.cochain({})
            for k in t.coeff_dict:
                deg=C.deg(C.cochain({k:1}))
                for i in range(len(P.basis)):
                    if self.vec_rep[i]!=0:
                        result+=self.vec_rep[i]*t.coeff_dict[k]*P.basis[i].cochain_ad_dicts[deg][k]
            return result
        raise ValueError('ad must only be applied to objects T_symb_elt, T_symb_basis_elt, ext_elt, and cochain')
        
    def iprod(self,other):
        '''other: another T_symb_elt or T_symb_basis_elt
           returns: the inner product of self and other'''
        if not hasattr(other,'parent'): raise ValueError('iprod recieved an invalid arg') 
        if self.parent!=other.parent: raise invalid_parent_exception
            
        return(sum([self.vec_rep[i]*other.vec_rep[i]*self.parent.iprod_list[i] 
                for i in range(len(self.parent.basis))]))
    
    def cast_as_ext_elt(self):
        result_dict=remove_zeros({(self.parent.basis_strs[i],):self.vec_rep[i] for i in range(len(self.vec_rep))})
        return self.parent.ext_alg.elt(result_dict)
    
    def cast_as_cochain(self):
        result_dict=remove_zeros({(self.parent.basis_strs[i],):self.vec_rep[i] for i in range(len(self.vec_rep))})
        return self.parent.cochain_complex.cochain(result_dict)


In [32]:
class T_symb:
    def __init__(self,heis_dim,pickled_ad=False):
        if heis_dim%2!=1 or heis_dim<1: raise heis_dim_exception
           
        self.heis_dim=heis_dim
        
        self.basis_strs=['Y','H','E','X']
        for i in range(1,heis_dim):
            self.basis_strs.append('e%d'%i)
        self.basis_strs.append('N')
        self.neg_basis_strs=self.basis_strs[3:len(self.basis_strs)+1]
        
        # Weights
        self.wght_list=[1,0,0,-1]
        for i in range(4,len(self.basis_strs)-1):
            self.wght_list.append(-i+3)
        self.wght_list.append(-heis_dim)
        
        # Bases
        self.basis=[T_symb_basis_elt(self.basis_strs[i],self.wght_list[i],self) 
                    for i in range(len(self.basis_strs))]
        self.gl2_basis=self.basis[0:4]
        self.heis_basis=self.basis[4:len(self.basis)]
        self.V_basis=self.heis_basis[0:len(self.heis_basis)-1]
        self.neg_basis=self.basis[3:len(self.basis)+1]
        
        self.cochain_complex=cochain_complex(self)
        self.ext_alg=ext_alg(self)
        
        self.ad_dict={}
        self.ad_mats=[]
        self.set_ad_dict(pickled_ad)
        self.set_dual_ad_dict()
        
        self.cochain_complex.init_ad_2_cochain()
        
        self.iprod_list=[1,2,2,1]+[factorial(i-1)/factorial(heis_dim-i-1) for i in range(1,heis_dim)]+[1]
        
    def set_ad_dict(self,pickled_ad=False):
        # Following Medvedev's basis/conventions, except the error in [Y,e_i]=(i-1)*(2*m+1-i)e_{i-1}
        k=len(self.basis)

        Y=self.gl2_basis[0]
        H=self.gl2_basis[1]
        E=self.gl2_basis[2]
        X=self.gl2_basis[3]
        N=self.heis_basis[len(self.heis_basis)-1]
        ## Set ad_dicts
        #  First, set ad_dict for each T_symb_basis_elt

        for A in self.gl2_basis:
            E.ad_dict[str(A)]=T_symb_elt([0]*k,self)
        for i in range(1,len(self.V_basis)+1):
            E.ad_dict[str(self.V_basis[i-1])]=self.V_basis[i-1]
        E.ad_dict['N']=2*N

        X.ad_dict['Y']=H
        X.ad_dict['H']=-2*X
        X.ad_dict['E']=T_symb_elt([0]*k,self)
        X.ad_dict['X']=T_symb_elt([0]*k,self)
        for i in range(1,len(self.V_basis)):
            X.ad_dict[str(self.V_basis[i-1])]=self.V_basis[i]
        X.ad_dict[str(self.V_basis[len(self.V_basis)-1])]=T_symb_elt([0]*k,self)
        X.ad_dict['N']=T_symb_elt([0]*k,self)

        Y.ad_dict['Y']=T_symb_elt([0]*k,self)
        Y.ad_dict['H']=2*Y
        Y.ad_dict['E']=T_symb_elt([0]*k,self)
        Y.ad_dict['X']=-H
        Y.ad_dict[str(self.V_basis[0])]=T_symb_elt([0]*k,self)
        for i in range(2,len(self.V_basis)+1):
            Y.ad_dict[str(self.V_basis[i-1])]=(i-1)*(self.heis_dim-i)*self.V_basis[i-2]
        Y.ad_dict['N']=T_symb_elt([0]*k,self)

        H.ad_dict['Y']=-2*Y
        H.ad_dict['H']=T_symb_elt([0]*k,self)
        H.ad_dict['E']=T_symb_elt([0]*k,self)
        H.ad_dict['X']=2*X
        for i in range(1,len(self.V_basis)+1):
            H.ad_dict[str(self.V_basis[i-1])]=(2*i-self.heis_dim)*self.V_basis[i-1]
        H.ad_dict['N']=T_symb_elt([0]*k,self)

        for i in range(1,len(self.V_basis)+1):
            if i>=2: self.V_basis[i-1].ad_dict['Y']=-(i-1)*(self.heis_dim-i)*self.V_basis[i-2]
            else: self.V_basis[i-1].ad_dict['Y']=T_symb_elt([0]*k,self)
            self.V_basis[i-1].ad_dict['H']=-(2*i-self.heis_dim)*self.V_basis[i-1]
            self.V_basis[i-1].ad_dict['E']=-self.V_basis[i-1]
            if i<=self.heis_dim-2: self.V_basis[i-1].ad_dict['X']=-self.V_basis[i]
            else: self.V_basis[i-1].ad_dict['X']=T_symb_elt([0]*k,self)
            for j in range(1, len(self.V_basis)+1):
                if i+j==self.heis_dim: self.V_basis[i-1].ad_dict[str(self.V_basis[j-1])]=(-1)**i*N
                else: self.V_basis[i-1].ad_dict[str(self.V_basis[j-1])]=T_symb_elt([0]*k,self)
            self.V_basis[i-1].ad_dict['N']=T_symb_elt([0]*k,self)

        N.ad_dict['Y']=T_symb_elt([0]*k,self)
        N.ad_dict['H']=T_symb_elt([0]*k,self)
        N.ad_dict['E']=-2*N
        N.ad_dict['X']=T_symb_elt([0]*k,self)
        for i in range(1,len(self.V_basis)+1):
            N.ad_dict[str(self.V_basis[i-1])]=T_symb_elt([0]*k,self)
        N.ad_dict['N']=T_symb_elt([0]*k,self)
        self.set_ad_mats()
    
    def set_ad_mats(self):
        for i in range(len(self.basis)):
            A=self.basis[i]
            r=Matrix([A.ad(B).vec_rep for B in self.basis])
            self.ad_mats.append(r.transpose())
            
    def Ad_mat(self,A):
        '''arg: a T_symb_elt or T_symb_basis_elt
           returns: the matrix rep of Ad(exp(A))'''
        ad_mat=zeros(len(self.basis))
        for i in range(len(self.basis)):
            ad_mat+=A.vec_rep[i]*self.ad_mats[i]
        return exp(ad_mat)
            
    
    def jacobi_test(self):
        '''returns: True if the Jacobi identity holds, False otherwise'''
        for A in self.basis:
            for B in self.basis:
                for C in self.basis:
                    t1=self.ad(A,self.ad(B,C))
                    t2=self.ad(self.ad(A,B),C)+self.ad(B,self.ad(A,C))
                    if t1!=t2: return False
        return True
    
    def set_dual_ad_dict(self):
        # reset the dicts
        for A in self.basis:
            A.dual_ad_dict={B:self.elt([0]*len(self.basis)) for B in self.basis_strs}
        # ad(A,B)=C ==> ad*(A,C*)-=B*
        for A in self.basis:
            for B in self.basis_strs:
                val=A.ad_dict[B]
                for i in range(len(self.basis)):
                    if val.vec_rep[i]!=0:
                        C=self.basis[i]
                        A.dual_ad_dict[str(C)]-=val.vec_rep[i]*self.basis[self.basis_strs.index(B)]
        
    
    def elt(self,vec_rep=None):
        if vec_rep==None: return(T_symb_elt([0]*len(self.basis),self))
        return T_symb_elt(vec_rep,self)
    
    def sort_basis_tuple(self,basis_tuple):
        '''basis_tuple: a tuple of str_reps of T_symb_basis_elt objs
           returns: a tuple containing an rearrangement of basis_tuple of descending degree, 
           and the sign of the permutation (either -1 or 1)'''
        basis_list=list(basis_tuple)
        sorted_list=basis_list.copy()
        sorted_list.sort(key=lambda A:self.basis_strs.index(A))
        return(tuple(sorted_list),permutation_sign(basis_list,sorted_list))
    
    
    def set_ext_ad_dict(self,deg,pickled=False):
        '''sets A.ext_ad_dicts[deg] for each A in self.basis'''
        # Don't reset a dict that's already been written
        if deg in self.basis[0].ext_ad_dicts: return
        self.ext_alg.init_basis(deg)
        
        if pickled:
            print('I should write some unpickling code here!')
            return
        
        # initialize if necessary
        if self.ext_alg==None: self.ext_alg=ext_alg(self)
        if deg not in self.ext_alg.basis:
            self.ext_alg.init_basis(deg)
            
        # construct the dict
        for A in self.basis:
            A.ext_ad_dicts[deg]={}
        
        # set the dict
        if deg==0:
            for A in self.basis:
                # ad(A,1)=0
                A.ext_ad_dicts[0]={tuple():0}
            return
        
        # set the previous dict
        #I'm not sure how to incorporate pickling into this call
        self.set_ext_ad_dict(deg-1)
        
        if deg==1:
            for A in self.basis:
                for B in self.basis:
                    ext_B=self.ext_alg.elt({(str(B),):1})
                    temp=A.dual_ad_dict[str(B)].vec_rep
                    A.ext_ad_dicts[1][(str(B),)]=self.ext_alg.elt({(str(self.basis[i]),):temp[i]
                                                        for i in range(len(self.basis))})
            return
        
        for A in self.basis:
            for B in self.ext_alg.basis_strs[deg]:
                B1=B[0:len(B)-1]
                ext_B1=self.ext_alg.elt({B1:1})
                B2=B[len(B)-1:len(B)]
                ext_B2=self.ext_alg.elt({B2:1})
                A.ext_ad_dicts[deg][B]=(A.ext_ad_dicts[deg-1][B1].wedge(ext_B2)
                                       +ext_B1.wedge(A.ext_ad_dicts[1][B2]))
                
    # To Do: finish writing this algorithm
    # (Copy pasted from the above)
    def set_cochain_ad_dict(self,deg,pickled=False):
        # Don't reset a dict that's already been written
        if deg in self.basis[0].cochain_ad_dicts: return
        
        if pickled:
            print('I should write some unpickling code here!')
            return
        
        # initialize if necessary
        if self.cochain_complex==None: init_cochain_complex(self)
        C=self.cochain_complex
        if deg not in C.basis:
            C.init_basis(deg)
            
        # construct the dict
        for A in self.basis:
            A.cochain_ad_dicts[deg]={}
        
        # set the dict
        if deg==0:
            for A in self.basis:
                for B in self.basis:
                    v=A.ad_dict[str(B)].vec_rep
                    A.cochain_ad_dicts[0][(str(B),)]=C.cochain({(self.basis_strs[i],):v[i]
                                                                for i in range(len(self.basis))})
            return
        
        self.set_ext_ad_dict(deg)
        for A in self.basis:
            for B1 in self.ext_alg.basis_strs[deg]:
                for B2 in self.basis:
                    k=B1+(str(B2),)
                    c_B2=C.cochain({(str(B2),):1})
                    v=A.ad_dict[str(B2)].vec_rep
                    c_ad_B2=C.cochain({(str(self.basis[i]),):v[i] for i in range(len(self.basis))})
                    t1=self.ext_alg.elt({B1:1}).wedge(c_ad_B2)
                    t2=A.ext_ad_dicts[deg][B1].wedge(c_B2)
                    A.cochain_ad_dicts[deg][k]=t1+t2
    
    def ad(self,t,c):
        '''t: a T_symb_elt object
           c: an object of type T_symb_basis_elt, T_symb_elt, ext_elt, or cochain
           returns: ad(t,c)'''
        if type(t)==T_symb_elt or type(t)==T_symb_basis_elt:
            return t.ad(c)
        
        if type(t)==int and t==0:
            if type(c)==cochain:
                return c.parent.cochain({})
            if type(c)==ext_elt:
                return c.parent.elt({})
            if type(c)==T_symb_elt or type(c)==T_symb_basis_elt:
                return c.parent.elt([0]*len(c.parent.basis))
        raise ValueError('The first argument of ad should be of type T_symb_elt or T_symb_basis_elt')
        
    def iprod(self,t1,t2):
        '''t1,t2: of the same type among T_symb_basis_elt, T_symb_elt, ext_elt, and cochain'''
        if not hasattr(t1,'parent') and hasattr(t2,'parent'):
            raise ValueException('arguments of iprod must have type T_symb_basis_elt, T_symb_elt, ext_elt, or cochain')
        
        if t1==0 or t2==0:
            return 0
        
        if t1.parent==t2.parent:
            return t1.iprod(t2)
        
        raise invalid_parent_exception('arguments of iprod must have the same parent')

In [24]:
class heis_dim_exception(Exception):
    '''Raised when the provided heis_dim is not odd'''
    pass

class invalid_parent_exception(Exception):
    '''Raised when the parent of an argument isn't what it should be'''  
    pass